In [1]:
import math
import numpy as np
import itertools

# OWEA Class

In [2]:
# Implementation of the algorithm from On Optimal Designs for Nonlinear Models: A General and Efficient Algorithm
class owea:
    # initialize class with a function calculating variance of a model (as a 2d numpy) and interest parameter gradient function (as a 2d numpy)
    # points_range is a list of 2-tuples (min, max) for each variable's range
    def __init__(self, points_ranges, interest_parameter_gradient, information_func, cur_design_size, initial_information = None, initial_design_size = None):
        # parameters to calculate information matrix
        self.parameter_gradient = interest_parameter_gradient
        self.get_information = information_func
        self.init_info = initial_information
        self.init_design_size = initial_design_size
        if cur_design_size is not None:
            self.cur_design_size = cur_design_size
        else:
            self.cur_design_size = 1
        if initial_information is not None and initial_design_size is not None:
            self.total_design_size = initial_design_size + self.cur_design_size
        else:
            self.total_design_size = self.cur_design_size
        # information matrix of theta for current support points and their design,
        # combined with initial design along with its inverse,
        # and variance of combined design for parameter of interest
        self.single_infos_theta = None
        self.cur_design_info_theta = None
        self.final_design_info_theta = None
        self.final_design_invinfo_theta = None
        self.final_design_var = None
        self.final_design_invvar = None
        # parameters to track design space and optimal design
        self.design_space = None
        self.support_points = None
        self.support_weights = None
        self.point_ranges = np.array(points_ranges)
        self.design_space_initialized = False
        self.support_points_initialized = False
    # returns list of strings of support points/weights
    def get_design(self):
        return(
            [
                f"{idx+1}: Point: {p}, Weight: {w}"
                for idx, (p, w) in enumerate(zip(self.support_points, self.support_weights))
            ]
            if (self.support_points is not None and self.support_weights is not None)
            else 'No Current Design'
        )
    # calculates the information matrix of theta at a point as a 2d numpy
    def calc_single_information(self, *args, **kwargs):
        return( self.get_information(*args, **kwargs) )
    # calculates the information of theta along with inverse and the variance of the parameters of interest for current design
    def update_design_information(self, update_single_infos = False):
        # calculate the overall information matrix for the current design (weighted sum of information at support points)
        if update_single_infos:
            self.single_infos_theta = [self.calc_single_information(point) for point in self.support_points]
        cur_design_info_theta = np.sum([point_w * point_info for point_w, point_info in zip(self.support_weights, self.single_infos_theta)], axis=0)
        # if initial design information is provided, combine it with the current design information as weighted average based on the design sizes
        if (self.init_info is not None) and (self.init_design_size is not None) and (self.cur_design_size is not None):
            final_design_info_theta = (self.init_design_size / self.total_design_size) * self.init_info + (self.cur_design_size / self.total_design_size) * cur_design_info_theta
        else:
            final_design_info_theta = cur_design_info_theta
        # calculate variance of parameters of interest and its inverse
        # force exact symmetry due to floating-point symmetry
        final_design_invinfo_theta = np.linalg.inv(final_design_info_theta)
        final_design_invinfo_theta = (final_design_invinfo_theta + final_design_invinfo_theta.T) / 2.0 
        final_design_var = self.parameter_gradient @ final_design_invinfo_theta @ self.parameter_gradient.T
        final_design_invvar = np.linalg.inv(final_design_var)
        # update class variables for information and variance
        self.cur_design_info_theta = cur_design_info_theta
        self.final_design_info_theta = final_design_info_theta
        self.final_design_invinfo_theta = final_design_invinfo_theta
        self.final_design_var = final_design_var
        self.final_design_invvar = (final_design_invvar + final_design_invvar.T) / 2.0
    # add a single support with 0.0 weight and its information matrix
    def add_support(self, point, weight=0.0):
        self.support_points.append(point)
        self.support_weights.append(weight)
        self.single_infos_theta.append(self.calc_single_information(point))
    # calculate the first and second derivatives of modified phi with respect to the weights of the support points except last support point
    def calc_derivatives(self, criteria_p):
        num_support = len(self.support_points) - 1
        support_der_m = self.single_infos_theta[-1]
        support_ders = [
            self.total_design_size * (self.single_infos_theta[idx] - support_der_m)
            for idx in range(num_support)
        ]
        main_first_ders = [
            -1 * self.parameter_gradient @ self.final_design_invinfo_theta @ support_ders[idx] @ self.final_design_invinfo_theta @ self.parameter_gradient.T
            for idx in range(num_support)
        ]
        main_second_ders = [
            [
                self.parameter_gradient @ (
                    self.final_design_invinfo_theta @ support_ders[idx_j] @ self.final_design_invinfo_theta @ support_ders[idx_i] @ self.final_design_invinfo_theta
                    + self.final_design_invinfo_theta @ support_ders[idx_i] @ self.final_design_invinfo_theta @ support_ders[idx_j] @ self.final_design_invinfo_theta           
                ) @ self.parameter_gradient.T
                for idx_j in range(num_support)
            ]
            for idx_i in range(num_support)
        ]
        if criteria_p == 0:
            first_ders = [
                np.trace( self.final_design_invvar @ main_first_ders[idx] )
                for idx in range(num_support)
            ]
            second_ders = [
                [
                    np.trace(
                        self.final_design_invvar @ main_second_ders[idx_i][idx_j]
                        - self.final_design_invvar @ main_first_ders[idx_j] @ self.final_design_invvar @ main_first_ders[idx_i]
                    )
                    for idx_j in range(num_support)
                ]
                for idx_i in range(num_support)
            ]
        elif criteria_p > 0:
            first_ders = [
                criteria_p * np.trace(
                    np.linalg.matrix_power(self.final_design_var, criteria_p - 1) @ main_first_ders[idx]
                )
                for idx in range(num_support)
            ]
            second_ders = [
                [
                    criteria_p * np.trace(
                        np.linalg.matrix_power(self.final_design_var, criteria_p - 1) @ main_second_ders[idx_i][idx_j]
                    )
                    for idx_j in range(num_support)
                ]
                for idx_i in range(num_support)
            ]
            if criteria_p >= 2:
                add_second_ders = [
                    [
                        criteria_p * np.trace(
                            np.sum(
                                [
                                    np.linalg.matrix_power(self.final_design_var, k) @ main_first_ders[idx_i] @ np.linalg.matrix_power(self.final_design_var, criteria_p-2-k) @ main_first_ders[idx_j]
                                    for k in range(criteria_p - 2)
                                ],
                                axis=0
                            )
                        )
                        for idx_j in range(num_support)
                    ]
                    for idx_i in range(num_support)
                ]
                second_ders = [
                    [
                        second_ders[idx_i][idx_j] + add_second_ders[idx_i][idx_j]
                        for idx_j in range(num_support)
                    ]
                    for idx_i in range(num_support)
                ]
        else:
            raise ValueError("Derivative Calculation: Invalid criteria_p value. Must be non-negative.")
        return( np.array(first_ders), (np.array(second_ders) + np.array(second_ders).T) / 2.0 )
    # initialize the design space as grid of (init_grid_size) number of points along each axis within each axis's range 
    def set_design_space(self, init_grid_size):
        axes_points = [np.linspace(r[0], r[1], init_grid_size) for r in self.point_ranges]
        self.design_space = [np.array(point) for point in itertools.product(*axes_points)]
        self.design_space_initialized = True
    # initialize support points/weights with (init_num_points) points with uniform weights
    def set_support_points(self, init_num_points, init_method = 'equidistant'):
        ranges_min = self.point_ranges[:, 0]
        ranges_max = self.point_ranges[:, 1]
        if init_method == 'uniform':
            points = np.random.uniform(low = ranges_min, high = ranges_max, size = (init_num_points, ranges_min.shape[0]))
            self.support_points = [points[k] for k in range(init_num_points)]
        else:
            # default is equidistant support point initialization
            t = np.linspace(0, 1, init_num_points)[:, np.newaxis]
            points = ranges_min + t * (ranges_max - ranges_min)
            self.support_points = [row for row in points]
        self.support_weights = [1/init_num_points] * init_num_points
        self.support_points_initialized = True
    # check general equivalence theorem condition for phi_p optimality and also returns index of design point with maximum phi_p directional derivative
    def check_equivalence(self, criteria_p, criteria_eps):
        cur_max_point_idx = None
        cur_max_dirder = -math.inf
        # loop through points in design space and find point with maximum directional derivative for phi_p optimality criteria
        for idx, point in enumerate(self.design_space):
            point_info = self.calc_single_information(point)
            Apart_mini = self.parameter_gradient @ self.final_design_invinfo_theta
            Apart = (self.cur_design_size / self.total_design_size) * Apart_mini @ (point_info - self.cur_design_info_theta) @ Apart_mini.T
            # check phi_p optimality condition for current design space for specific p optimality
            if criteria_p == 0:
                cur_dirder = np.trace( self.final_design_invvar @ Apart )
            elif criteria_p > 0:
                first_part = np.trace( np.linalg.matrix_power(self.final_design_var, criteria_p) ) ** (1/criteria_p - 1)
                second_part = np.trace( np.linalg.matrix_power(self.final_design_var, criteria_p - 1) @ Apart )
                cur_dirder = first_part * second_part
            else:
                raise ValueError("General Equivalence Checking: Invalid criteria_p value. Must be non-negative.")
            # update point with maximum directional derivative if current point has higher directional derivative than previous max
            if cur_dirder > cur_max_dirder:
                cur_max_dirder = cur_dirder
                cur_max_point_idx = idx
        return( cur_max_dirder <= criteria_eps, cur_max_point_idx )
    # run the OWEA algorithm
    # verbose: True for printing which support points are added and removed
    def fit(self, support_init_method, init_num_points, init_grid_size, criteria_p, criteria_eps, algo_der_eps, algo_step_size_min, verbose = True):
        # initialize design space and support points/weights if needed
        if not self.design_space_initialized:
            self.set_design_space(init_grid_size)
        if not self.support_points_initialized:
            self.set_support_points(init_num_points, support_init_method)
        self.update_design_information(True)
        # update design and stop if general equivalence condition is satisfied
        equivalence_status, max_point_idx = self.check_equivalence(criteria_p, criteria_eps)
        while not equivalence_status:
            # find optimal weights for current support points using newton method
            newton_done = False
            alpha = 1
            init_weights = np.array(self.support_weights[:-1])
            first_ders, second_ders = self.calc_derivatives(criteria_p)
            second_ders_inv = np.linalg.inv(second_ders)
            while not newton_done:
                cur_weights = init_weights - alpha * second_ders_inv @ first_ders
                # if any new weights are negative, use a smaller update step size
                if np.any(cur_weights < 0) or np.sum(cur_weights) >= 1:
                    alpha = alpha / 2
                    # if step size becomes too small, remove support point with smallest weight (note these are boundary points)
                    if alpha < algo_step_size_min:
                        idx_min_weight = self.support_weights.index(min(self.support_weights))
                        del_point = self.support_points.pop(idx_min_weight)
                        del_weight = self.support_weights.pop(idx_min_weight)
                        del self.single_infos_theta[idx_min_weight]
                        if verbose:
                            print( f"Removed support ({del_point},{del_weight})" )
                        # recalculate all the weights, derivatives, and information
                        alpha = 1
                        self.update_design_information(False)
                        init_weights = np.array(self.support_weights[:-1])
                        first_ders, second_ders = self.calc_derivatives(criteria_p)
                        second_ders_inv = np.linalg.inv(second_ders)
                else:
                    # if all new weights are non-negative, update support weights
                    self.support_weights = cur_weights.tolist() + [1 - np.sum(cur_weights)]
                    alpha = 1
                    self.update_design_information(False)
                    init_weights = np.array(self.support_weights[:-1])
                    first_ders, second_ders = self.calc_derivatives(criteria_p)
                    second_ders_inv = np.linalg.inv(second_ders)
                    # (1) if optimal, newton method is done (2) if not optimal, continue newton at next iteration
                    if np.linalg.norm(first_ders) < algo_der_eps:
                        newton_done = True
            # check equivalence condition for set of support points
            # if condition fails, add point with maximum directional derivative to support points
            equivalence_status, max_point_idx = self.check_equivalence(criteria_p, criteria_eps)
            if not equivalence_status:
                self.add_support(self.design_space[max_point_idx], 0.0)
                self.update_design_information(False)
                if verbose:
                    print( f"Added to support {self.design_space[max_point_idx]}" )


# Example 1

In [3]:
# information matrix for point x in Example 1
theta = [0.1, 1, 0.2, 2]
def information_func(x):
    deta_dtheta = [
        np.exp(-1 * theta[1] * x),
        -1 * theta[0] * x * np.exp(-1 * theta[1] * x),
        np.exp(-1 * theta[3] * x),
        -1 * theta[2] * x * np.exp(-1 * theta[3] * x)
    ]
    deta_dtheta = np.array(deta_dtheta).reshape(-1,1)
    return( deta_dtheta @ deta_dtheta.T )

In [4]:
# single-stage design
ex1 = owea(
    points_ranges = [[0,3]],
    interest_parameter_gradient = np.eye(4),
    information_func = information_func,
    cur_design_size = 80
)

In [5]:
# criteria_p = 0 for D, criteria_p = 1 for A
ex1.fit(
    support_init_method = 'uniform',
    init_num_points = 6,
    init_grid_size = 1000,
    criteria_p = 0,
    criteria_eps = 0.01,
    algo_der_eps = 0.01,
    algo_step_size_min = 1e-5,
    verbose = True
)

Removed support ([1.29437349],2.1809223895919837e-05)
Removed support ([1.15816278],1.2857052462427105e-07)
Added to support [0.]
Added to support [2.94294294]
Removed support ([2.44383817],1.8227199187114517e-09)
Added to support [1.06306306]
Removed support ([0.84385166],9.346970500734112e-08)
Added to support [2.69069069]
Removed support ([2.94294294],3.1502721251730365e-08)
Added to support [1.17417417]
Removed support ([1.29286486],2.4011216362434355e-07)
Added to support [2.78978979]
Removed support ([2.69069069],2.4179050825650905e-08)
Added to support [1.11711712]
Removed support ([1.06306306],2.013926714103556e-06)


In [6]:
ex1.get_design()

['1: Point: [0.31868828], Weight: 0.24988837892739305',
 '2: Point: [0.], Weight: 0.24999194575392322',
 '3: Point: [1.17417417], Weight: 0.07300724400353735',
 '4: Point: [2.78978979], Weight: 0.24987253543789253',
 '5: Point: [1.11711712], Weight: 0.17723989587725386']

In [7]:
# multi-stage design
ex1 = owea(
    points_ranges = [[0,3]],
    interest_parameter_gradient = np.eye(4),
    information_func = information_func,
    cur_design_size = 80,
    initial_information = (information_func(0) + information_func(1) + information_func(2) + information_func(3)) / 4,
    initial_design_size = 40
)

In [8]:
# criteria_p = 0 for D, criteria_p = 1 for A
ex1.fit(
    support_init_method = 'uniform',
    init_num_points = 6,
    init_grid_size = 1000,
    criteria_p = 0,
    criteria_eps = 0.01,
    algo_der_eps = 0.01,
    algo_step_size_min = 1e-5,
    verbose = True
)

Removed support ([2.7483198],1.4275843017656646e-07)
Removed support ([0.87241367],1.603007840781056e-08)
Added to support [0.31831832]
Added to support [0.]
Removed support ([0.08136887],9.957677602556251e-09)
Added to support [1.17117117]
Removed support ([1.74950709],1.0769730668265942e-08)
Removed support ([0.77368968],7.681467961445434e-09)
Added to support [1.09309309]


In [9]:
ex1.get_design()

['1: Point: [2.86200367], Weight: 0.17269547417989134',
 '2: Point: [0.31831832], Weight: 0.3630522481121455',
 '3: Point: [0.], Weight: 0.2430372094832089',
 '4: Point: [1.17117117], Weight: 0.0523245983216503',
 '5: Point: [1.09309309], Weight: 0.16889046990310397']

# Example 2

In [10]:
# information matrix for point x in Example 2
theta = [1, 2, 0.1, -1, 0.2]
def information_func(x):
    deta_dtheta = [
        1,
        x[0],
        x[0] ** 2,
        x[1],
        x[0] * x[1]
    ]
    deta_dtheta = np.array(deta_dtheta).reshape(-1,1)
    return( deta_dtheta @ deta_dtheta.T )

In [11]:
# D-optimality with all parameters as interest
ex2 = owea(
    points_ranges = [[-1,1], [0,1]],
    interest_parameter_gradient = np.eye(5),
    information_func = information_func,
    cur_design_size = 120,
    initial_information = None,
    initial_design_size = None
)

In [12]:
# criteria_p = 0 for D, criteria_p = 1 for A
ex2.fit(
    support_init_method = 'uniform',
    init_num_points = 10,
    init_grid_size = 100,
    criteria_p = 1,
    criteria_eps = 0.01,
    algo_der_eps = 0.01,
    algo_step_size_min = 1e-5,
    verbose = True
)

Removed support ([-0.43174878  0.4734537 ],1.3789814334863593e-07)
Removed support ([-0.36914636  0.49676914],2.961486477521824e-08)
Removed support ([-0.1479803   0.13802978],1.264163280582006e-08)
Added to support [1. 1.]
Removed support ([0.80800829 0.55217019],1.2796917821296535e-08)
Removed support ([-0.45304156  0.75896096],1.8291224093253914e-09)
Added to support [-1.  0.]
Removed support ([-0.80237089  0.40949726],1.6318358747366765e-08)
Added to support [1. 0.]
Removed support ([0.77212826 0.35062526],2.583809354117421e-08)
Added to support [-1.  1.]
Removed support ([-0.84143293  0.67692403],3.04347694008476e-08)
Added to support [0.03030303 1.        ]
Removed support ([-0.19899055  0.76176471],1.7658710143095476e-08)
Added to support [0.13131313 0.        ]
Added to support [-0.09090909  0.        ]
Removed support ([-0.21340008  0.03985101],4.373684886095339e-07)
Added to support [0.01010101 0.        ]
Removed support ([0.13131313 0.        ],2.3635882782436954e-08)
Remov

In [13]:
ex2.get_design()

['1: Point: [1. 1.], Weight: 0.14014206496599294',
 '2: Point: [-1.  0.], Weight: 0.1857198048724806',
 '3: Point: [1. 0.], Weight: 0.18608509000752668',
 '4: Point: [-1.  1.], Weight: 0.13965860962715407',
 '5: Point: [0.01010101 0.        ], Weight: 0.10086654229587184',
 '6: Point: [-0.01010101  1.        ], Weight: 0.11966111452083278',
 '7: Point: [-0.01010101  0.        ], Weight: 0.1278667737101411']

# Example 4.1

In [14]:
# information matrix for point x in Example 1
theta = [1, 0.5, 1, 1]
def information_func(x):
    deta_dtheta = [
        np.exp(theta[1] * x),
        theta[0] * x * np.exp(theta[1] * x),
        np.exp(theta[3] * x),
        theta[2] * x * np.exp(theta[3] * x)
    ]
    deta_dtheta = np.array(deta_dtheta).reshape(-1,1)
    return( deta_dtheta @ deta_dtheta.T )
# g(theta) = theta_1 * theta_2 * exp(theta_2 x) + theta_3 * theta_4 * exp(theta_4 x)
# consider x = 0
gradient_mat = np.array([theta[1], theta[0], theta[3], theta[2]]).reshape(1,4)

In [15]:
# single-stage design
ex4 = owea(
    points_ranges = [[0,1]],
    interest_parameter_gradient = gradient_mat,
    information_func = information_func,
    cur_design_size = 100
)

In [16]:
# criteria_p = 0 for D, criteria_p = 1 for A
ex4.fit(
    support_init_method = 'uniform',
    init_num_points = 6,
    init_grid_size = 100,
    criteria_p = 0,
    criteria_eps = 0.01,
    algo_der_eps = 0.01,
    algo_step_size_min = 1e-5,
    verbose = True
)

Removed support ([0.39381686],1.8765016207566987e-09)
Removed support ([0.57290981],7.716235890405998e-08)
Added to support [1.]
Removed support ([1.],0.0)
Added to support [1.]
Removed support ([1.],0.0)
Added to support [1.]
Removed support ([1.],0.0)
Added to support [1.]
Removed support ([0.61964291],7.767832606525929e-09)
Added to support [0.80808081]
Removed support ([0.55596059],8.918957898129447e-09)
Added to support [0.35353535]
Removed support ([0.11256767],8.007732477966958e-08)
Added to support [0.]
Removed support ([0.04939458],1.4381894688449617e-07)
Added to support [0.3030303]
Removed support ([0.35353535],8.307687491905288e-07)
Added to support [0.78787879]
Removed support ([0.80808081],5.742852118929993e-06)


In [17]:
ex4.get_design()

['1: Point: [1.], Weight: 0.05517702723662357',
 '2: Point: [0.], Weight: 0.34981132208975785',
 '3: Point: [0.3030303], Weight: 0.4448914124527169',
 '4: Point: [0.78787879], Weight: 0.1501202382209017']

# Example 4.2

In [21]:
# information matrix for point x in Example 1
theta = [1, 0.5, 1, 1]
def information_func(x):
    deta_dtheta = [
        1 / (x + theta[1]),
        -1 * theta[0] / ((x + theta[1]) ** 2),
        1 / (x + theta[3]),
        -1 * theta[2] / ((x + theta[3]) ** 2)
    ]
    deta_dtheta = np.array(deta_dtheta).reshape(-1,1)
    return( deta_dtheta @ deta_dtheta.T )
# g(theta) = -theta_1 / (x + theta_2)^2 - theta_3 / (x + theta_4)^2
# consider x = 0
gradient_mat = np.array(
    [
        -1 / (theta[1] ** 2),
        2 * theta[0] / (theta[1] ** 3),
        -1 / (theta[3] ** 2),
        2 * theta[2] / (theta[3] ** 3),
    ]
).reshape(1,4)

In [22]:
# single-stage design
ex4 = owea(
    points_ranges = [[0,1]],
    interest_parameter_gradient = gradient_mat,
    information_func = information_func,
    cur_design_size = 100
)

In [23]:
# criteria_p = 0 for D, criteria_p = 1 for A
ex4.fit(
    support_init_method = 'uniform',
    init_num_points = 7,
    init_grid_size = 100,
    criteria_p = 1,
    criteria_eps = 0.01,
    algo_der_eps = 0.01,
    algo_step_size_min = 1e-5,
    verbose = True
)

Removed support ([0.65671791],1.1441916175081168e-06)
Removed support ([0.60818806],1.7634687332623855e-07)
Removed support ([0.37891766],1.8077184416043224e-07)
Added to support [1.]
Removed support ([0.75222123],3.721655676012076e-08)
Added to support [0.11111111]
Removed support ([0.24405613],4.717835593038956e-08)
Added to support [0.]
Removed support ([0.01235537],5.3284944555676694e-08)
Added to support [0.47474747]
Removed support ([0.54376763],2.539864457017798e-07)
Added to support [0.09090909]
Removed support ([0.11111111],1.5051866739435727e-06)


In [24]:
ex4.get_design()

['1: Point: [1.], Weight: 0.058014499255289724',
 '2: Point: [0.], Weight: 0.3584870597750217',
 '3: Point: [0.47474747], Weight: 0.13997393914234363',
 '4: Point: [0.09090909], Weight: 0.44352450182734493']

# Example HW2

In [25]:
# information matrix for point x in HW2
theta = [1, 2]
def information_func(x):
    val = np.exp(theta[0] + theta[1] * x)
    deta_dtheta = [
        (val / ((1 + val) ** 2)),
        (val / ((1 + val) ** 2)) * x
    ]
    deta_dtheta = np.array(deta_dtheta).reshape(-1,1)
    return( deta_dtheta @ deta_dtheta.T )

In [26]:
# single-stage design
ex = owea(
    points_ranges = [[-10,10]],
    interest_parameter_gradient = np.eye(2),
    information_func = information_func,
    cur_design_size = 100
)

In [27]:
# criteria_p = 0 for D, criteria_p = 1 for A
ex.fit(
    support_init_method = 'uniform',
    init_num_points = 4,
    init_grid_size = 10000,
    criteria_p = 0,
    criteria_eps = 0.01,
    algo_der_eps = 0.01,
    algo_step_size_min = 1e-5,
    verbose = True
)

Removed support ([-9.23583249],1.1758963935903921e-07)
Removed support ([-7.99413503],1.2149584194541643e-09)
Added to support [-0.3570357]
Removed support ([6.61034553],1.2399149484920713e-09)
Added to support [-1.19111911]
Removed support ([-3.8567482],3.7498037054237675e-08)
Added to support [0.1090109]
Added to support [-0.93709371]
Removed support ([-0.3570357],7.164038786059215e-07)
Removed support ([-1.19111911],1.2466318103775184e-08)
Added to support [-1.02910291]
Added to support [-0.0150015]
Removed support ([-0.93709371],5.967691017049055e-07)
Removed support ([0.1090109],8.669716870505223e-09)


In [28]:
ex.get_design()

['1: Point: [-1.02910291], Weight: 0.5000124168504863',
 '2: Point: [-0.0150015], Weight: 0.49998758314951375']